## Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import datetime
import einops
from inductive_bias.MLP_metrics import measure_metrics
import copy
from inductive_bias.local_cache import CacheContext
from tyche.math import gaussint_ln_noncentral_erf
from tyche.math import gaussint_ln_riemann
from tyche.utils import weighted_logsumexp
from torch import nn
import torch as t
from typing import List
from jaxtyping import Float, Int
from inductive_bias.MLP_init import *
from inductive_bias.MLP_metrics import run_train_and_estimator
from inductive_bias.inductive_bias_viz import generate_heatmaps
import itertools
import plotly.graph_objs as go  # Import the graph objects from Plotly
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from tqdm.notebook import tqdm
import pandas as pd
from tyche.estimator import VolumeConfig, VolumeEstimator

In [3]:
device = t.device("cuda:7" if t.cuda.is_available() else "cpu")
t.set_default_device(device)

## Evaluating hyperparameter search

In [10]:
WEIGHT_MODES = ["uniform", "xavier_uniform", "normal", "xavier_normal", "constant"]

In [ ]:
path = "/home/louis/tyche/scripts/shared_database_3.parquet"
df_new = pd.read_parquet(path)
WEIGHT_MODES = ["none"]
plots = generate_heatmaps(df=df_new, weight_mode="none", cmap="coolwarm")

In [ ]:
path = "/home/louis/tyche/scripts/shared_database_10.parquet"
df_old = pd.read_parquet(path)
WEIGHT_MODES = ["none"]
plots = generate_heatmaps(df=df_old, weight_mode="none", cmap="coolwarm")

## training attempt

In [53]:
params = MLPConfig(
    activation=t.nn.ReLU(),
    N=53,
    embed_dimension=36,
    linear_dimension=48,
    intermediate="pure",
    embedding_tied=False,
    unembedding_tied=False,
    bias_unembed=True,
    bias_layer=True,
    num_additional_layers=5,
    dimensions=(48),
    W_amplitude=1,
    weight_mode="none",
    device=device,
    b_amplitude=0.0,
    train_data_size=1600,
    training_epochs=0,
    eval_interval=1000,
    weight_decay=2e-4,
    save_model=False,
)
model = MLP_VARIANTS(params)
model.initialize_weights()

volume_cfg = VolumeConfig(
    model_type="mlp",
    model=model,
    n_samples=100,
    iters=15,
    cutoff=1e-2,
    cache_mode=None,
    chunking=False,
    reduction=None,
    tol=0.0351,
    tqdm=False,
    gaussint_fn=gaussint_ln_riemann,
    new_estimator=False,
)

with CacheContext(cache_dir=f".cache/{volume_cfg.new_estimator}"):
    results = run_train_and_estimator(
        custom_config=params, gpu_id=7, volume_config=volume_cfg
    )

/home/louis/tyche/.venv/lib/python3.13/site-packages/torch/utils/_device.py:104: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)
0it [00:00, ?it/s]


In [51]:
results[0]["volume_estimates"]

array([5.7490451e+08, 5.7490470e+08, 5.7490451e+08, 5.7490419e+08,
       5.7490445e+08, 5.7490426e+08, 5.7490432e+08, 5.7490438e+08,
       5.7490458e+08, 5.7490451e+08, 5.7490464e+08, 5.7490445e+08,
       5.7490426e+08, 5.7490438e+08, 5.7490451e+08, 5.7490458e+08,
       5.7490470e+08, 5.7490464e+08, 5.7490470e+08, 5.7490470e+08,
       5.7490451e+08, 5.7490445e+08, 5.7490413e+08, 5.7490426e+08,
       5.7490451e+08, 5.7490426e+08, 5.7490432e+08, 5.7490438e+08,
       5.7490458e+08, 5.7490438e+08, 5.7490477e+08, 5.7490451e+08,
       5.7490445e+08, 5.7490451e+08, 5.7490451e+08, 5.7490470e+08,
       5.7490477e+08, 5.7490451e+08, 5.7490426e+08, 5.7490445e+08,
       5.7490426e+08, 5.7490458e+08, 5.7490458e+08, 5.7490458e+08,
       5.7490445e+08, 5.7490438e+08, 5.7490445e+08, 5.7490432e+08,
       5.7490458e+08, 5.7490464e+08, 5.7490438e+08, 5.7490426e+08,
       5.7490419e+08, 5.7490451e+08, 5.7490432e+08, 5.7490419e+08,
       5.7490445e+08, 5.7490451e+08, 5.7490432e+08, 5.7490458e

In [54]:
import os


folder = "/home/louis/tyche/notebooks/.cache/True"
subfolders = [f.path for f in os.scandir(folder)]
cache_outputs = [
    t.load(subfolder)["output"] for subfolder in subfolders if subfolder.endswith(".pt")
]

In [58]:
folder = "/home/louis/tyche/notebooks/.cache/False"
subfolders = [f.path for f in os.scandir(folder)]
cache_outputs_old = [
    t.load(subfolder)["output"] for subfolder in subfolders if subfolder.endswith(".pt")
]

In [59]:
cache_outputs_old

[tensor([[33108.4727]], device='cuda:7'),
 tensor([[32926.7305]], device='cuda:7'),
 tensor([[33152.4414]], device='cuda:7'),
 tensor([[33430.9062]], device='cuda:7'),
 tensor([[33185.1836]], device='cuda:7'),
 tensor([[33358.7578]], device='cuda:7'),
 tensor([[33351.9844]], device='cuda:7'),
 tensor([[33271.4297]], device='cuda:7'),
 tensor([[33078.9492]], device='cuda:7'),
 tensor([[33117.4648]], device='cuda:7'),
 tensor([[32988.3242]], device='cuda:7'),
 tensor([[33213.3398]], device='cuda:7'),
 tensor([[33312.5703]], device='cuda:7'),
 tensor([[33271.8828]], device='cuda:7'),
 tensor([[33163.6406]], device='cuda:7'),
 tensor([[32954.7148]], device='cuda:7'),
 tensor([[32953.8672]], device='cuda:7'),
 tensor([[33011.7930]], device='cuda:7'),
 tensor([[32972.7695]], device='cuda:7'),
 tensor([[32920.9336]], device='cuda:7'),
 tensor([[33136.9609]], device='cuda:7'),
 tensor([[33190.0234]], device='cuda:7'),
 tensor([[33424.0312]], device='cuda:7'),
 tensor([[33367.8672]], device='cu

In [57]:
cache_outputs

[{'input': {'args': (),
   'kwargs': {'a': tensor(136.4855, device='cuda:7'),
    'b': tensor(2.9401, device='cuda:7'),
    'n': 21628,
    'x1': tensor([[15.5000]], device='cuda:7'),
    'c': tensor(-10814.5000, device='cuda:7'),
    'debug': False,
    'interval_count': 10000.0,
    'riemann_eps': 1e-30,
    'tol': 0.0351,
    'y_tol': 5,
    'iters': 15,
    'allow_unconverged': False,
    'init_mult': 1,
    'rtol': 0.1},
   'timestamp': 1746043548.466576,
   'function': 'gaussint_ln_riemann'},
  'output': tensor([[33182.4922]], device='cuda:7')},
 {'input': {'args': (),
   'kwargs': {'a': tensor(136.4855, device='cuda:7'),
    'b': tensor(17.4254, device='cuda:7'),
    'n': 21628,
    'x1': tensor([[14.5000]], device='cuda:7'),
    'c': tensor(-10814.5000, device='cuda:7'),
    'debug': False,
    'interval_count': 10000.0,
    'riemann_eps': 1e-30,
    'tol': 0.0351,
    'y_tol': 5,
    'iters': 15,
    'allow_unconverged': False,
    'init_mult': 1,
    'rtol': 0.1},
   'timesta

In [20]:
for cache_old, cache_new in zip(cache_outputs_old, cache_outputs):
    if cache_old[0] == cache_new[0] and cache_old[1] == cache_new[1]:
        print("Same cache output")
    else:
        print("Difference")
        break

Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache output
Same cache

In [15]:
cache_outputs

[(tensor([[15.5000]], device='cuda:7'), tensor([0.0093], device='cuda:7')),
 (tensor([[14.5000]], device='cuda:7'), tensor([0.0109], device='cuda:7')),
 (tensor([[14.5000]], device='cuda:7'), tensor([0.0103], device='cuda:7')),
 (tensor([[32.]], device='cuda:7'), tensor([0.0094], device='cuda:7')),
 (tensor([[17.5000]], device='cuda:7'), tensor([0.0095], device='cuda:7')),
 (tensor([[12.5000]], device='cuda:7'), tensor([0.0102], device='cuda:7')),
 (tensor([[17.5000]], device='cuda:7'), tensor([0.0091], device='cuda:7')),
 (tensor([[32.]], device='cuda:7'), tensor([0.0092], device='cuda:7')),
 (tensor([[13.]], device='cuda:7'), tensor([0.0102], device='cuda:7')),
 (tensor([[32.]], device='cuda:7'), tensor([0.0094], device='cuda:7')),
 (tensor([[32.]], device='cuda:7'), tensor([0.0094], device='cuda:7')),
 (tensor([[32.]], device='cuda:7'), tensor([0.0098], device='cuda:7')),
 (tensor([[12.]], device='cuda:7'), tensor([0.0106], device='cuda:7')),
 (tensor([[14.5000]], device='cuda:7'), 

In [5]:
path = "/home/louis/tyche/notebooks/.cache/debugFalse/find_radius_vectorized_41d07ffc527824038b7a79a2fdbc8bb1.pt"
t.load(path)

RuntimeError: PytorchStreamReader failed locating file data.pkl: file not found

In [22]:
x = t.ones(params.N, device=device)

In [23]:
y = t.ones(params.N) * 2

In [26]:
(y * x * y).sum()

tensor(212., device='cuda:7')

In [38]:
results = run_train_and_estimator(
    custom_config=params, gpu_id=7, volume_config=volume_cfg
)

TypeError: get_estimates_vectorized_gauss() missing 1 required positional argument: 'n'

In [30]:
results = run_train_and_estimator(
    custom_config=params, gpu_id=7, volume_config=volume_cfg
)

tensor([-2.9401], device='cuda:7')
tensor([-17.4254], device='cuda:7')
tensor([0.5487], device='cuda:7')
tensor([22.6577], device='cuda:7')
tensor([3.1630], device='cuda:7')
tensor([17.2882], device='cuda:7')
tensor([16.3759], device='cuda:7')
tensor([10.0491], device='cuda:7')
tensor([-5.3021], device='cuda:7')
tensor([-2.1676], device='cuda:7')
tensor([-12.4482], device='cuda:7')
tensor([5.4451], device='cuda:7')
tensor([18.1806], device='cuda:7')
tensor([10.0224], device='cuda:7')
tensor([1.4434], device='cuda:7')
tensor([-11.6552], device='cuda:7')
tensor([-15.2467], device='cuda:7')
tensor([-10.6405], device='cuda:7')
tensor([-13.6885], device='cuda:7')
tensor([-17.8857], device='cuda:7')
tensor([-0.6672], device='cuda:7')
tensor([3.5473], device='cuda:7')
tensor([27.4731], device='cuda:7')
tensor([17.6179], device='cuda:7')
tensor([1.9135], device='cuda:7')
tensor([21.4322], device='cuda:7')
tensor([13.9058], device='cuda:7')
tensor([10.4322], device='cuda:7')
tensor([-6.6952], d

0it [00:00, ?it/s]


In [19]:
save_dir = f"/mnt/ssd-1/louis/inductive_bias"

In [7]:
# show subfolders of save_dir
subfolders = [f.path for f in os.scandir(save_dir) if f.is_dir()]

In [ ]:
subfolders

In [9]:
m = t.load(subfolders[0] + "/model_0.pt", map_location=device)

In [ ]:
# load m into model
model.load_state_dict(m)

In [11]:
import os
import torch as t
import einops
import plotly.graph_objects as go


def plot_indicator_table(model, params, save=False):
    device = next(model.parameters()).device
    N = params.N
    group_set = [[i, j, (i + j) % N] for i in range(N) for j in range(N)]
    inputs = t.tensor([g[:2] for g in group_set], dtype=t.long).to(device)

    with t.no_grad():
        model.eval()
        logits = model(inputs)  # shape N^2 x N
        max_prob_entry = t.argmax(logits, dim=-1)  # shape N^2

    output_matrix = einops.rearrange(max_prob_entry, "(n m) -> n m", n=N)  # shape N x N
    hover_labels = [[f"{output_matrix[j][i]}" for i in range(N)] for j in range(N)]
    row_labels = [str(g) for g in range(N)]
    col_labels = row_labels

    # Generate N different colors for the heatmap
    import plotly.colors as pc

    # Using a colorscale that works well for categorical data
    if N <= 10:
        # For small N, use distinct colors from the Plotly qualitative colorscales
        colors = pc.qualitative.Plotly[:N]
    else:
        # For larger N, generate a continuous colorscale with N distinct colors
        colorscale = pc.sequential.Viridis
        colors = [pc.sample_colorscale(colorscale, i / (N - 1))[0] for i in range(N)]

    # Create the colorscale with proper scaling between 0 and 1
    custom_colorscale = []
    for i in range(N):
        # Lower bound for this color
        custom_colorscale.append([i / N, colors[i]])
        # Upper bound for this color (except for the last color)
        if i < N - 1:
            custom_colorscale.append([(i + 1) / N, colors[i]])

    fig = go.Figure(
        data=go.Heatmap(
            z=output_matrix.tolist(),
            showscale=False,
            colorscale=custom_colorscale,
            x=col_labels,
            y=row_labels,
            zmin=0,
            zmax=N - 1,
            customdata=hover_labels,
            hovertemplate="x=%{x}<br>"
            + "y=%{y}<br>"
            + "z=%{customdata}<extra></extra>",
        ),
    )

    fig.update_layout(
        title=f"Final run",
        xaxis={
            "showgrid": True,
            "side": "top",
            "ticks": "outside",
            "tickmode": "array",
            "tickvals": [i for i in range(N)],
            "ticktext": row_labels,
        },
        yaxis={
            "showgrid": True,
            # "autorange": "reversed",
            "side": "left",
            "ticks": "outside",
            "tickmode": "array",
            "tickvals": [i for i in range(N)],
            "ticktext": col_labels,
        },
        height=900,
        width=900,
    )

    if save:
        # Create plots directory if it doesn't exist
        if not os.path.exists("plots"):
            os.mkdir("plots")
        fig.write_html("./plots/plot_final.html")

    return fig

In [ ]:
plot_indicator_table(model=model, params=params, save=True)

## Investigate local similarity

In [ ]:
ACTS = [t.nn.ReLU(), t.nn.GELU(), t.nn.Tanh()]

result_acts = {}
models_acts = []
from tyche.MLP_metrics import run_train_and_estimator

for act in ACTS:
    params = MLPConfig(
        activation=act,
        N=53,
        embed_dimension=36,
        linear_dimension=48,
        intermediate="pure",
        embedding_tied=False,
        unembedding_tied=False,
        bias_unembed=False,
        num_additional_layers=2,
        dimensions=(48),
        bias_layer=False,
        W_amplitude=np.sqrt(10) ** -0.5,
        weight_mode="xavier_uniform",
        device=device,
        b_amplitude=0.0,
        train_data_size=53**2,
        training_epochs=0,
        seed=1,
    )

    model = MLP_VARIANTS(params)
    model.initialize_weights()
    break
    models_acts.append(model)

    with CacheContext(f".cache/run_{act.__class__.__name__}"):
        result_dict = run_train_and_estimator(custom_config=params, gpu_id=7)[0]

    # result_dict = run_train_and_estimator(custom_config=params, gpu_id=7)[0]

    result_acts[f"{act}"] = result_dict

In [ ]:
from tyche.volume import VolumeResult

# Register it as a safe class with PyTorch
import torch.serialization

torch.serialization.add_safe_globals([VolumeResult])
cache_dict = {}
for act in ACTS:
    folder_dir = f".cache/run_{act.__class__.__name__}"

    subfolders = [f.path for f in os.scandir(folder_dir)]
    print(act)
    tensors = [
        t.load(subfolder, weights_only=False)
        for subfolder in subfolders
        if subfolder.endswith(".pt")
    ]

    cache_dict[f"{act}"] = tensors

In [ ]:
cache_dict["GELU(approximate='none')"][0]["input"]

In [ ]:
cache_dict["Tanh()"][0]

In [ ]:
for k, v in cache_dict.items():
    diff = v - cache_dict["ReLU()"]
    print(diff)

In [ ]:
cache_dict["ReLU()"][:10]

In [ ]:
cache_dict["Tanh()"][:10]

In [155]:
volume_estimates = [result_acts[act]["volume_estimates"] for act in result_acts.keys()]

In [139]:
for v in volume_estimates:
    diff = v - volume_estimates[0]
    print((diff**2).sum())

In [ ]:
models_acts[0]

In [141]:
model_vec = t.nn.utils.parameters_to_vector(model.parameters())
new_vec = model_vec + 0
new_model = copy.deepcopy(model)
t.nn.utils.vector_to_parameters(new_vec, new_model.parameters())

In [157]:
d_model = model_vec.shape[0]

In [156]:
def random_direction_model(
    model: MLP_VARIANTS,
    direction: Float[t.Tensor, "d_params"],
    mult: float = 1.0,
    dataset: Optional[Int[t.Tensor, "train_set_size 2"]] = None,
):

    if dataset is None:
        dataset = t.tensor(
            list(itertools.product(range(model.mlp_config.N), repeat=2)),
            device=model.mlp_config.device,
        )

    model_vec = t.nn.utils.parameters_to_vector(model.parameters())
    new_vec = model_vec + direction * mult
    new_model = copy.deepcopy(model)
    t.nn.utils.vector_to_parameters(new_vec, new_model.parameters())

    logits = model(dataset)
    new_logits = new_model(dataset)

    probs = t.nn.functional.softmax(logits, dim=-1)
    logprobs_new = t.nn.functional.log_softmax(new_logits, dim=-1)

    kl_div = (
        t.nn.functional.kl_div(logprobs_new, probs, reduction="none").sum(dim=-1).mean()
    )

    return kl_div

In [144]:
direction = t.randn(d_model, device=device)
direction = direction / direction.norm()
kl_divs = {f"{model.mlp_config.activation}": [] for model in models_acts}

In [ ]:
direction = t.randn(d_model, device=device)
direction.sum()

In [158]:
# different random_seed
direction = t.randn(d_model, device=device)
direction = direction / direction.norm()
kl_divs = {f"{model.mlp_config.activation}": [] for model in models_acts}
for model in models_acts:
    for mult in np.arange(0, 100, 0.5):
        kl_div = random_direction_model(model, direction, mult=mult)
        kl_divs[f"{model.mlp_config.activation}"].append(kl_div.item())

In [ ]:
print(kl_divs["ReLU()"][-1])

In [160]:
kl_nats = 1e-2

# Convert to bits
kl_bits = kl_nats * np.log2(np.e)

In [ ]:
for k, v in kl_divs.items():
    plt.plot(v[:50], label=k)
plt.axhline(y=kl_nats, color="red", linestyle="--", label="y=1e-2")
plt.xlabel("Mult")
plt.ylabel("KL Divergence")
plt.legend()
plt.title("KL Divergence for different activations")
plt.show()

## Complexity over time

In [6]:
path = "/home/louis/tyche/scripts/shared_database_True_2809.parquet"
df = pd.read_parquet(path)
epoch_list = df["epoch"].unique()

In [53]:
df_epoch = df[df["epoch"] == 5000].copy()

In [60]:
mask = (
    (df_epoch["activation"] == "ReLU")
    & (df_epoch["num_additional_layers"] == 5)
    & (df_epoch["intermediate"] == "pure")
)

In [66]:
df_specific = df_epoch[mask]

hashable_columns = []
for col in df_specific.columns:
    try:
        num_unique = df_specific[col].nunique()
        if num_unique > 1:
            hashable_columns.append(col)
    except TypeError:  # This will catch the unhashable type error
        pass  # Skip columns with unhashable types

filtered_df = df_specific[hashable_columns]

In [ ]:
filtered_df[filtered_df["W_amplitude"] == 1.0]

In [ ]:
df_epoch = df[df["epoch"] == 10000].copy()
generate_heatmaps(
    df=df_epoch,
    weight_mode="none",
    stat_type="mean",
    value_stat="mean",
    figsize=(15, 10),
    cmap="coolwarm",
    value_column="test_loss",
)

In [ ]:
figs_dict = {
    "figs_volume": [],
    "figs_test_loss": [],
    "figs_test_accuracy": [],
    "figs_train_loss": [],
    "figs_train_accuracy": [],
}
for epoch in epoch_list:
    mask = df["epoch"] == epoch
    df_epoch = df[mask]
    fig_volume = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
    )
    figs_dict["figs_volume"].append(fig_volume)
    fig_test_loss = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="test_loss",
    )
    figs_dict["figs_test_loss"].append(fig_test_loss)
    fig_test_accuracy = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="test_accuracy",
    )
    figs_dict["figs_test_accuracy"].append(fig_test_accuracy)
    fig_train_loss = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="train_loss",
    )
    figs_dict["figs_train_loss"].append(fig_train_loss)
    fig_train_accuracy = generate_heatmaps(
        df=df_epoch,
        weight_mode="none",
        stat_type="mean",
        value_stat="mean",
        figsize=(15, 10),
        cmap="coolwarm",
        value_column="train_accuracy",
    )
    figs_dict["figs_train_accuracy"].append(fig_train_accuracy)

In [ ]:
figs_dict

In [31]:
epoch_list = [epoch.item() for epoch in df["epoch"].unique()]

In [ ]:
from inductive_bias.inductive_bias_viz import export_figure_dict_to_html


interactive_figure = export_figure_dict_to_html(
    figure_dict=figs_dict, epoch_list=epoch_list
)

In [20]:
df_test = df[df["epoch"] == 0].copy()

In [ ]:
df_test["activation"].unique()

In [ ]:
# ploy df_test
generate_heatmaps(
    df=df_test,
    weight_mode="none",
    stat_type="mean",
    volume_stat="mean",
    cmap="coolwarm",
    title="epoch 0",
)

## Gaussian integral


In [6]:
from tyche.volume import VolumeResult

# Register it as a safe class with PyTorch
import torch.serialization

torch.serialization.add_safe_globals([VolumeResult])
ACTS = [t.nn.ReLU(), t.nn.GELU(), t.nn.Tanh()]
cache_dict = {}
for act in ACTS:
    folder_dir = f".cache/run_{act.__class__.__name__}"

    subfolders = [f.path for f in os.scandir(folder_dir)]

    tensors = [
        t.load(subfolder, weights_only=False)
        for subfolder in subfolders
        if subfolder.endswith(".pt")
    ]

    cache_dict[f"{act.__class__.__name__}"] = tensors

input = cache_dict["ReLU"][0]["input"]["kwargs"]
input_test = copy.deepcopy(input)

## Testing new integral with Pythia

In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

from tyche import VolumeConfig, VolumeEstimator


# Load any CausalLM model, tokenizer, and dataset
model = AutoModelForCausalLM.from_pretrained("EleutherAI/pythia-14m")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-14m")
tokenizer.pad_token_id = 1  # pythia-specific
tokenizer.eos_token_id = 0  # pythia-specific
dataset = load_dataset(
    "EleutherAI/lambada_openai", name="en", split="test", trust_remote_code=True
)

# Configure the estimator
cfg = VolumeConfig(
    model=model,
    tokenizer=tokenizer,
    dataset=dataset,
    text_key="text",  # must match dataset field
    n_samples=100,  # number of MC samples
    cutoff=1e-2,  # KL-divergence cutoff (nats)
    max_seq_len=2048,  # max sequence length for tokenizer or chunk_and_tokenize
    val_size=10,  # number of dataset sequences to use. default (None) uses all.
    cache_mode=None,  # see below
    chunking=False,  # whether to use chunk_and_tokenize
    gaussint_fn=gaussint_ln_riemann,
)
estimator = VolumeEstimator.from_config(cfg)


# Run the estimator

tokens.shape=torch.Size([10, 224])


In [7]:
with CacheContext(f".cache/run_{model.__class__.__name__}"):
    result = estimator.run()

  0%|          | 0/100 [00:00<?, ?it/s]

tensor([1.], device='cuda:0')


  1%|          | 1/100 [00:01<02:15,  1.36s/it]

tensor([1.], device='cuda:0')


  2%|▏         | 2/100 [00:02<01:49,  1.11s/it]

...

  3%|▎         | 3/100 [00:03<01:33,  1.03it/s]

tensor([1.], device='cuda:0')


  4%|▍         | 4/100 [00:04<01:37,  1.02s/it]


tensor([1.0000], device='cuda:0')


KeyboardInterrupt: 

In [ ]:
result.estimates.mean().item()

In [12]:
folder_dir = ".cache/pythia"
subfolders = [f.path for f in os.scandir(folder_dir)]

caches = []

for subfolder in subfolders:
    if subfolder.endswith(".pt"):
        cache = t.load(subfolder, weights_only=False)
        caches.append(cache)

In [ ]:
caches[0]["output"].shape

In [8]:
a = t.load(
    "/home/louis/tyche/notebooks/.cache/pythia_riemann/gaussint_ln_riemann_0ae1081069a169a0a2532cf285b88068.pt"
)

In [ ]:
a["output"].unsqueeze(dim=0).unsqueeze(dim=0)

In [ ]:
from tyche.math import gaussint_ln_riemann


outputs_riemann = []
outputs_laplace = []

for cache in caches:

    input_dict = cache["input"]["kwargs"]
    output_riemann = gaussint_ln_riemann(**input_dict)
    outputs_riemann.append(output_riemann.item())
    outputs_laplace.append(cache["output"].item())

In [ ]:
int(1e6)

In [ ]:
outputs_laplace

In [64]:
diff = np.array(outputs_riemann) - np.array(outputs_laplace)

In [ ]:
diff / np.array(outputs_laplace)

In [ ]:
outputs_laplace

In [ ]:
torch.linspace(1e-30, input_dict["x1"].item(), int(1e6))

In [ ]:
input_dict["x1"].item()